In [1]:
import pandas as pd
import numpy as np

In [2]:
sold = pd.read_csv('/Users/morganstevenson/Desktop/IDX/week 6/sold_final.csv')
listings = pd.read_csv('/Users/morganstevenson/Desktop/IDX/week 6/listings_final.csv')

/var/folders/ch/xc77p32n37v6pfqh60_ls62c0000gn/T/ipykernel_57696/1813371982.py:1: DtypeWarning: Columns (0: ListAgentEmail, 1: BuyerAgencyCompensationType) have mixed types. Specify dtype option on import or set low_memory=False.
  sold = pd.read_csv('/Users/morganstevenson/Desktop/IDX/week 6/sold_final.csv')
/var/folders/ch/xc77p32n37v6pfqh60_ls62c0000gn/T/ipykernel_57696/1813371982.py:2: DtypeWarning: Columns (0: ListAgentEmail, 1: BuyerAgencyCompensationType) have mixed types. Specify dtype option on import or set low_memory=False.
  listings = pd.read_csv('/Users/morganstevenson/Desktop/IDX/week 6/listings_final.csv')


In [3]:
def add_iqr_flag(df, column, multiplier=1.5):

    Q1 = df[column].quantile(0.25)
    Q3 = df[column].quantile(0.75)
    IQR = Q3 - Q1

    lower = Q1 - multiplier * IQR
    upper = Q3 + multiplier * IQR

    flag_col = f"{column}_IQRFlag"

    df[flag_col] = (
        (df[column] < lower) |
        (df[column] > upper)
    )

    return df

In [4]:
sold = add_iqr_flag(sold, "ClosePrice")
sold = add_iqr_flag(sold, "LivingArea")
sold = add_iqr_flag(sold, "PricePerSqFt")
sold = add_iqr_flag(sold, "LotSizeSquareFeet")

In [5]:
listings = add_iqr_flag(listings, "ClosePrice")
listings = add_iqr_flag(listings, "LivingArea")
listings = add_iqr_flag(listings, "PricePerSqFt")
listings = add_iqr_flag(listings, "LotSizeSquareFeet")

In [6]:
def compare_iqr_filter(df, columns):
    summary = []
    filtered_dfs = {}

    for col in columns:
        flag_col = f"{col}_IQRFlag"

        filtered = df[~df[flag_col]].copy()
        filtered_dfs[col] = filtered

        summary.append({
            "Column": col,
            "Original Size": len(df),
            "Filtered Size": len(filtered),
            "Rows Removed": len(df) - len(filtered),
            "Original Median": df[col].median(),
            "Filtered Median": filtered[col].median()
        })

    summary_df = pd.DataFrame(summary)

    return summary_df, filtered_dfs

In [9]:
cols = [
    "ClosePrice",
    "LivingArea",
    "LotSizeSquareFeet",
    "PricePerSqFt"
]

comparison, filtered_sold = compare_iqr_filter(sold, cols)

comparison

,Column,Original Size,Filtered Size,Rows Removed,Original Median,Filtered Median
0,ClosePrice,422138,391440,30698,820000.000000,780000.000000
1,LivingArea,422138,403873,18265,1644.000000,1606.000000
2,LotSizeSquareFeet,422138,361293,60845,7256.000000,6600.000000
3,PricePerSqFt,422138,404858,17280,534.633683,521.255061


In [10]:
cols = [
    "ClosePrice",
    "LivingArea",
    "LotSizeSquareFeet",
    "PricePerSqFt"
]

comparison, filtered_listings = compare_iqr_filter(listings, cols)

comparison

,Column,Original Size,Filtered Size,Rows Removed,Original Median,Filtered Median
0,ClosePrice,136828,128335,8493,825000.00000,790000.000000
1,LivingArea,136828,131105,5723,1622.00000,1589.000000
2,LotSizeSquareFeet,136828,117262,19566,7128.00000,6534.000000
3,PricePerSqFt,136828,131803,5025,542.00542,529.026217


In [13]:
sold_flag_cols = [col for col in sold.columns if col.endswith("_IQRFlag")]

sold_filtered = sold.loc[
    ~sold[sold_flag_cols].any(axis=1)
].copy()

In [14]:
listing_flag_cols = [col for col in listings.columns if col.endswith("_IQRFlag")]

listings_filtered = listings.loc[
    ~listings[listing_flag_cols].any(axis=1)
].copy()

In [15]:
print(f"Sold: {len(sold):,} → {len(sold_filtered):,}")
print(f"Listings: {len(listings):,} → {len(listings_filtered):,}")

Sold: 422,138 → 326,511
Listings: 136,828 → 107,137


In [18]:
sold.to_csv('sold_flagged.csv', index= False)
sold_filtered.to_csv('sold_filtered.csv', index= False)
listings.to_csv('listings_flagged.csv', index= False)
listings_filtered.to_csv('listings_filtered.csv', index= False)